In [4]:
# 9_cluster_values_k_means.ipynb
#
# Produces national-level personas using flat K-Means on UKHLS feature data.
#
#   • K-Means is fit ONCE on ALL UKHLS respondents (4_feature_eng/{WAVE}_feature_eng.pkl)
#   • A single flat clustering is used — tribe labels ('1', '2', '3', …) are
#     consistent across any downstream LA-level join.
#   • Output: one DNA-profile row per cluster, written to OUTPUT_CSV.
#
# Pipeline:
#   Phase 1 — Fit national clusters (UKHLS feature pickle, ~19K rows):
#     Normalise features and run K-Means over all respondents.
#     Store: pidp → tribe_label  (cached as parquet).
#
#   Phase 2 — Profile clusters from UKHLS data directly:
#     Build DNA rows from the in-memory UKHLS dataframe using the fitted labels.
#     Append to output CSV.
#
# Output: data/8_cluster_national_level/LA_national_clusters.csv

import sys, os, gc
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_pipeline.config.config_paths as _cp
importlib.reload(_cp)
from data_pipeline.config.config_paths import DATA_FOLDER
import data_pipeline.config_variables as _cv
import data_pipeline.config_cluster   as _cc
_cv.reload_config_variables()
importlib.reload(_cc)

# Re-import after reload so names reflect the current config_variables.py on disk.
_cv_fresh = __import__('data_pipeline.config_variables', fromlist=['*'])
CLUSTER_VARS     = _cv_fresh.CLUSTER_VARS
SUMMARY_VARS     = _cv_fresh.SUMMARY_VARS
VARIABLE_MAP     = _cv_fresh.VARIABLE_MAP
CATEGORICAL_VARS = _cv_fresh.CATEGORICAL_VARS
CATEGORY_MAPS    = _cv_fresh.CATEGORY_MAPS
CONTINUOUS_VARS  = _cv_fresh.CONTINUOUS_VARS
expected_cluster_feature_columns = _cv_fresh.expected_cluster_feature_columns
print(f'SUMMARY_VARS has {len(SUMMARY_VARS)} variables: {SUMMARY_VARS}')

import pandas as pd
import numpy  as np
from pathlib import Path

import data_pipeline.helpers.normalise as normalise
import data_pipeline.helpers.cluster as cf
importlib.reload(normalise)
importlib.reload(cf)

from data_pipeline.config_cluster import WAVE, N_CLUSTERS

# ── Config ────────────────────────────────────────────────────────────────────
FEATURE_PKL    = f"../{DATA_FOLDER}/4_feature_eng/{WAVE}_feature_eng.pkl"
OUTPUT_DIR     = Path(f"../{DATA_FOLDER}/8_cluster_national_level")

if not os.path.exists(FEATURE_PKL):
    raise FileNotFoundError(
        f"{FEATURE_PKL} not found — run 4_feature_eng_ukhls "
        "(and optionally KEEP/5a if your CLUSTER_VARS use derived columns)."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "national_clusters.csv"
if OUTPUT_CSV.exists():
    OUTPUT_CSV.unlink()
    print(f"Removed existing {OUTPUT_CSV.name}")

ASSIGN_CACHE = OUTPUT_DIR / f"{WAVE}_national_pidp_assignments.parquet"


def _national_assign_cache_stale() -> bool:
    if not ASSIGN_CACHE.exists():
        return True
    try:
        return os.path.getmtime(FEATURE_PKL) > os.path.getmtime(ASSIGN_CACHE)
    except OSError:
        return True


# ── Phase 1: Fit national clusters on ALL UKHLS respondents ──────────────────
df_ukhls = pd.read_pickle(FEATURE_PKL)
df_ukhls['pidp'] = df_ukhls['pidp'].astype('int64')
print(f"Loaded {len(df_ukhls):,} UKHLS respondents × {len(df_ukhls.columns)} cols")

if _national_assign_cache_stale():
    print(f"\n── Phase 1: fitting clusters (N_CLUSTERS={N_CLUSTERS}) ──")

    _expected_feat = expected_cluster_feature_columns(WAVE)
    feature_cols  = [c for c in _expected_feat if c in df_ukhls.columns]
    missing_feat  = [c for c in _expected_feat if c not in df_ukhls.columns]
    print(f"  Feature columns: {len(feature_cols)} present, {len(missing_feat)} missing")
    if missing_feat:
        print(f"    Missing: {missing_feat}")

    avail_feat = [c for c in feature_cols if df_ukhls[c].notna().any()]

    coeffs  = normalise.fit(df_ukhls, avail_feat)
    df_norm = normalise.apply(df_ukhls, coeffs)

    n        = len(df_norm)
    labels   = cf.fit_kmeans(df_norm[avail_feat].values, N_CLUSTERS)

    print(f"  {n:,} respondents → {N_CLUSTERS} cluster(s)")

    df_assign = pd.DataFrame({
        'pidp':        df_ukhls['pidp'].values,
        'tribe_label': [str(lbl + 1) for lbl in labels],
    })
    df_assign['pidp'] = df_assign['pidp'].astype('int64')
    print(f"  {df_assign['tribe_label'].nunique()} national tribes assigned")

    df_assign.to_parquet(ASSIGN_CACHE, index=False)
    print(f"  Cached → {ASSIGN_CACHE}")

else:
    print(f"\n── Phase 1: skipped (cache up-to-date) ──")
    df_assign = pd.read_parquet(ASSIGN_CACHE)
    df_assign["pidp"] = df_assign["pidp"].astype("int64")
    print(f"  Loaded {len(df_assign):,} assignments from {ASSIGN_CACHE.name}")

# ── Phase 2: Profile clusters from UKHLS data ────────────────────────────────
print(f"\n── Phase 2: building DNA profiles from UKHLS data ──")

df_profiling = df_ukhls.merge(df_assign, on='pidp', how='inner')

rows = []
for tribe_label, sub_df in df_profiling.groupby('tribe_label', sort=False):
    row = cf.build_dna_row(
        tribe_label, sub_df, WAVE,
        SUMMARY_VARS, VARIABLE_MAP, CATEGORICAL_VARS, CATEGORY_MAPS,
        continuous_vars=CONTINUOUS_VARS,
    )
    row['cluster_level'] = 'national'
    rows.append(row)

result = pd.DataFrame(rows).sort_values('tribe_label')
result.to_csv(OUTPUT_CSV, index=False)
print(f"Written {len(result)} tribe rows to {OUTPUT_CSV}")
print(result[['cluster_level', 'tribe_label', 'size']].to_string(index=False))

api_clusters_dir = Path("../api/data/clusters")
api_clusters_dir.mkdir(parents=True, exist_ok=True)
import shutil
api_copy = api_clusters_dir / OUTPUT_CSV.name
shutil.copy2(OUTPUT_CSV, api_copy)
print(f"Copied to {api_copy}")


SUMMARY_VARS has 9 variables: ['doby_dv', 'sex_dv', 'racel_dv', 'hiqual_dv', 'jbstat', 'marstat_dv', 'tenure_dv', 'hhtype_dv', 'scsf1']
Loaded 27,330 UKHLS respondents × 67 cols

── Phase 1: fitting clusters (N_CLUSTERS=10) ──
  Feature columns: 53 present, 0 missing
  27,330 respondents → 10 cluster(s)
  10 national tribes assigned
  Cached → ../data/8_cluster_national_level/k_national_pidp_assignments.parquet

── Phase 2: building DNA profiles from UKHLS data ──
Written 10 tribe rows to ../data/8_cluster_national_level/national_clusters.csv
cluster_level tribe_label  size
     national           1  7844
     national          10   218
     national           2  1348
     national           3  6614
     national           4  2427
     national           5   375
     national           6  1699
     national           7  1702
     national           8    42
     national           9  5061
Copied to ../api/data/clusters/national_clusters.csv
